# Candidate Generation: Feature-Aware NCF (Early and Late Fusion)

Two feature-aware neural candidate generators that combine ID embeddings with request and article features:

- **Early fusion** encodes the user and article sides independently, then scores the concatenation of the tower outputs.
- **Late fusion** fuses the user and article ID embeddings first, then appends the request and article features before the final head.

Protocol:

- Build positives from session clicks in the full `labels_train` pool and negatives from exposed non-clicks in the same slate.
- Redraw up to four exposed negatives per positive from the same slate on every epoch.
- Train BPR with AdamW, selecting the epoch by NDCG on a chronological 90% cut of the training period.
- Refit on the full training period for the selected number of epochs.
- Score each request over articles with `available_from <= timestamp`, excluding the request's `full_history_rows`.
- Break ties by one-hour-lagged click popularity descending, then ascending `news_code`; backfill short lists from popularity.
- Write the standard `impression_uid, cutoff, news_code` table and evaluate it with `cg_evaluation.py`.

The models are static: they are fitted once on the training split and never updated with target-period interactions. User features are pooled `text64` and `entity100` history vectors, article features are the category one-hot and article `text64`, and popularity is not a model feature. `EMBED_DIM = 256`, the tower/head shapes and the 500-candidate output are fixed in this notebook.


## 0. Configure

**This is the only cell you need to edit.** Set `REUSE_ARTIFACTS = False` to retrain the model
and rebuild every artifact. Candidate lists are always regenerated for `TARGET_SPLIT`, so a new
validation or test split is served normally.


In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer.
ENVIRONMENT = "local"  # "local" or "colab"

# ---- Paths -------------------------------------------------------------------------------------
# Data path: the prepared data. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# On Colab, the zipped data in <submission>/2.Feature_Engineering is unpacked into the data
# path on the first run (a few minutes); later runs reuse it.
COLAB_DATA_PATH = None  # used on Colab; None = the project folder (4.Code)
LOCAL_DATA_PATH = None  # used locally; None = the project folder

# ---- Experiment ------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "validation"

# True: load the saved model checkpoint and skip training. Serving still runs, so candidate
# lists are always regenerated for TARGET_SPLIT.
# False: retrain the model from scratch.
# Baselines without a model reuse any cached intermediate state, but their candidate lists are
# still regenerated.
REUSE_ARTIFACTS = True

# ---- Reused artifacts when REUSE_ARTIFACTS=True -----------------------------------------------
# Loaded checkpoints, so training is skipped:
#   artifacts/ncf_features_early_bpr/model.pt
#   artifacts/ncf_features_late_bpr/model.pt
# Candidate lists are always regenerated.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn


if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(
        DATA_PATH
        or os.environ.get("NEWSREC_DATA_ROOT")
        or os.environ.get("NEWSREC_ROOT")
        or CODE_ROOT
    )
    .expanduser()
    .resolve()
)
REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
SUBMISSION_ROOT = CODE_ROOT.parent if ENVIRONMENT == "colab" else None
BUNDLE_DIR = SUBMISSION_ROOT / "2.Feature_Engineering" if SUBMISSION_ROOT else None
if BUNDLE_DIR is not None and BUNDLE_DIR.is_dir():
    import zipfile

    for archive in sorted(BUNDLE_DIR.glob("*.zip")):
        marker = DATA_ROOT / f".unpacked_{archive.stem}"
        if not marker.exists():
            print(f"Unpacking {archive.name} into {DATA_ROOT} (first run only) ...")
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(DATA_ROOT)
            marker.touch()

missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT
CG_DIR = CODE_ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(CODE_ROOT / "src"))

import itertools

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

from newsrec.common import Settings

PROCESSED = DATA_ROOT / "1_Data_Preparation" / "processed"
FEATURES = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"
ARTIFACTS = (
    CODE_ROOT.parent / "3.Model_Artifacts" / "CG"
    if ENVIRONMENT == "colab"
    else DATA_ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
)
RESULTS = DATA_ROOT / "results"

FIT_SPLIT = "train"

TRIALS = ("late", "early")
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)

EMBED_DIM = 256
TEXT_DIM = 64
ENTITY_DIM = 100
HISTORY_DIM = TEXT_DIM + ENTITY_DIM
HOUR_NS = 3_600 * 10**9
POPULARITY_SPLITS = ["train", "validation"]
BRANCH_DIMS = (512, 256)
EARLY_HEAD_DIMS = (256, 128, 64)
LATE_ID_DIMS = (256, 128, 64)
LATE_HEAD_DIMS = (256, 128, 64)
DROPOUT = 0.2
NUM_EPOCHS = 50
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SCORE_PAIRS = 131_072
SEED = 4222
RUN_LOSSES = ("bpr",)
LOSS = RUN_LOSSES[0]
PATIENCE = 3
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print("Trials:", ", ".join(TRIALS))
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 2. Data ingestion

Each request profile is summarised causally from its `full_history_rows` as the mean of non-zero `text64` and `entity100` article vectors, L2-normalised to 164 dimensions. Article features are a 17-way category one-hot plus `text64` (81 dimensions). The hourly popularity matrix uses training and validation click counts read one hour before the request, so it is causal within the target split, and it feeds only tie-breaking and fallback, never the model. Positives are session clicks from the full `labels_train` pool; each epoch redraws up to four exposed non-clicks from the same slate, restricted to the chronological fit window for tuning and applied to all rows for the refit. Tuning slates are sampled deterministically from `candidates_train` and NDCG uses the deepest slate length as `k`.


In [2]:
context = load_context(DATA_ROOT)
n_users, n_items = context.n_users, context.n_items

article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code", "category"],
).sort("article_row")

TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

CATEGORY_VOCAB = sorted(article_index["category"].unique().to_list())
CATEGORY_DIM = len(CATEGORY_VOCAB)
category_code = {value: index for index, value in enumerate(CATEGORY_VOCAB)}
article_onehot = np.zeros((n_items, CATEGORY_DIM), dtype=np.float32)
article_onehot[
    np.arange(1, n_items),
    [category_code[value] for value in article_index["category"]],
] = 1.0

USER_EXTRA_DIM = HISTORY_DIM
ITEM_EXTRA_DIM = CATEGORY_DIM + TEXT_DIM


def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return (
        pl.read_parquet(
            parts,
            columns=[
                "impression_uid",
                "user_code",
                "timestamp",
                "full_history_rows",
            ],
        )
        .sort("timestamp", maintain_order=True)
        .with_row_index("profile_row")
    )


def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n, dim = len(history_rows), matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        if lengths.sum() == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue
        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        valid_vectors = vectors[valid]
        starts = np.r_[0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1]
        pooled = np.add.reduceat(valid_vectors, starts, axis=0)
        pooled /= np.diff(np.r_[starts, len(valid_groups)])[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result


def build_features(profiles):
    history_rows = profiles["full_history_rows"].to_list()
    features = np.zeros((len(history_rows), HISTORY_DIM), dtype=np.float32)
    features[:, :TEXT_DIM] = pool_profiles(history_rows, TEXT)
    features[:, TEXT_DIM:] = pool_profiles(history_rows, ENTITY)
    return features


train_profiles = load_profiles(FIT_SPLIT)
train_features = build_features(train_profiles)
target_profiles = load_profiles(TARGET_SPLIT)
target_features = build_features(target_profiles)

print(f"Users: {n_users:,}")
print(f"Articles: {n_items - 1:,}")
print(f"Categories: {CATEGORY_DIM}")
print(f"User extra dim: {USER_EXTRA_DIM}")
print(f"Item extra dim: {ITEM_EXTRA_DIM}")
print(f"Training requests: {train_profiles.height:,}")
print(f"Target requests: {target_profiles.height:,}")

Users: 50,001
Articles: 51,282
Categories: 17
User extra dim: 164
Item extra dim: 81
Training requests: 95,071
Target requests: 31,624


In [3]:
index_codes = article_index["news_code"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = article_index["article_row"].to_numpy().astype(np.int64)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                (pl.col("timestamp").dt.epoch("ns") // HOUR_NS).alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(pl.col("clicked").cast(pl.Int64).sum())
    .collect()
)

MIN_HOUR = int(hourly["hour"].min())
MAX_HOUR = int(hourly["hour"].max())
N_HOURS = MAX_HOUR - MIN_HOUR + 1
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS[
    hourly["hour"].to_numpy().astype(np.int64) - MIN_HOUR,
    row_of_code[hourly["news_code"].to_numpy().astype(np.int64)],
] = hourly["clicked"].to_numpy().astype(np.int32)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def hourly_popularity(hour_index, item_rows):
    clk = CLICKS[hour_index - 1, item_rows].astype(np.float32)
    return np.log1p(clk)


ITEM_EXTRA = np.concatenate(
    [article_onehot, np.asarray(TEXT, dtype=np.float32)], axis=1
)
ITEM_EXTRA_TENSOR = torch.from_numpy(ITEM_EXTRA).to(DEVICE)

catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, n_items, dtype=np.int32),
        "news_code": context.news_code[1:].astype(np.int32),
        "available_from": context.available_from[1:],
    }
).sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = catalogue["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = catalogue["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = catalogue["available_from"].to_numpy().astype("datetime64[ns]")
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(ARTICLE_ROWS_SORTED.size, dtype=np.int64)

print("Item extra:", ITEM_EXTRA.shape)
print("Catalogue:", len(ARTICLE_ROWS_SORTED))
print(f"Rolling popularity hours: {N_HOURS:,}")

Item extra: (51283, 81)
Catalogue: 51282
Rolling popularity hours: 120


In [4]:
run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if (
    TRAINING_ROWS["clicked"].isna().any()
    or not TRAINING_ROWS["clicked"].isin([0, 1]).all()
):
    raise ValueError("Training rows must be uniquely binary-labelled")

profile_lookup = train_profiles.select(
    "impression_uid", "timestamp", "profile_row", "user_code"
)
train_user_codes = train_profiles["user_code"].to_numpy().astype(np.int64)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = profile_lookup.filter(pl.col("impression_uid").is_in(positive_uids))
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[TRAINING_ROWS["impression_uid"].isin(fit_uids)].reset_index(
    drop=True
)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build feature-NCF arrays."""
    groups, _ = EPOCH_SAMPLER.draw(int(epoch), uids=fit_uids if fit_only else None)
    groups = (
        pl.from_pandas(groups)
        .join(profile_lookup, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if (
        groups["profile_row"].null_count()
        or groups["user_code"].null_count()
        or groups["pos_row"].null_count()
    ):
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes", empty_as_null=True)
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)] = (
        negative_long["neg_row"].to_numpy().astype(np.int64)
    )
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups["user_code"].to_numpy().astype(np.int64),
        groups["profile_row"].to_numpy().astype(np.int64),
        groups["pos_row"].to_numpy().astype(np.int64),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "profile_row"),
    max_impressions=None,
    seed=SEED + 1,
)

TUNE_PROFILE_IDX = tune["profile_row"].to_numpy().astype(np.int64)
TUNE_USER_IDS = train_user_codes[TUNE_PROFILE_IDX]
TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int64)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
TUNE_UIDS = tune["impression_uid"].to_numpy()

print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")
print(f"Tuning rows: {tune.height:,}")
print(f"Tuning NDCG k: {TUNE_NDCG_K}")

Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning rows: 343,011
Tuning NDCG k: 31


## 3. Model

Early fusion runs a user tower over `256` ID dimensions plus `164` request features (`420 -> 512 -> 256`) and an item tower over `256` ID dimensions plus `81` article features (`337 -> 512 -> 256`), then scores the concatenated branches (`512 -> 256 -> 128 -> 64 -> 1`). Late fusion first projects the concatenated user and article ID embeddings (`512 -> 256 -> 128 -> 64`), then appends the request and article features before the head (`309 -> 256 -> 128 -> 64 -> 1`). Both models use GELU, dropout `0.2` and a zeroed padding row `0`; the output is a ranking score, not a calibrated click probability.

Architecture:

```text
EARLY FUSION
user_code ------> user_embedding (256) --+
user features  (164) --------------------+--> concat (420) -> 512 -> 256 -> u_branch
article_row ----> item_embedding (256) --+
item features   (81) --------------------+--> concat (337) -> 512 -> 256 -> v_branch
concat [u_branch, v_branch] (512) -> 256 -> 128 -> 64 -> 1

LATE FUSION
user_code ------> user_embedding (256) --+
                                         +--> concat (512) -> 256 -> 128 -> 64 -> fused
article_row ----> item_embedding (256) --+
concat [fused, user features (164), item features (81)] (309) -> 256 -> 128 -> 64 -> 1

Both models: GELU + dropout 0.2. BPR objective (one positive vs up to 4 exposed negatives per step).
```


In [5]:
def mlp(dims, dropout=DROPOUT, final_activation=True):
    layers = []
    for index, (in_dim, out_dim) in enumerate(itertools.pairwise(dims)):
        layers.append(nn.Linear(in_dim, out_dim))
        if final_activation or index < len(dims) - 2:
            layers.extend([nn.GELU(), nn.Dropout(dropout)])
    return nn.Sequential(*layers)


class EarlyFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        branch_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.branch_dim = branch_dims[-1]
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.user_tower = mlp([embed_dim + user_extra_dim, *branch_dims], dropout)
        self.item_tower = mlp([embed_dim + item_extra_dim, *branch_dims], dropout)
        self.head = mlp(
            [2 * self.branch_dim, *head_dims, 1], dropout, final_activation=False
        )

    def encode_users(self, user_ids, user_extra):
        return self.user_tower(
            self.input_dropout(
                torch.cat([self.user_embedding(user_ids), user_extra], dim=-1)
            )
        )

    def encode_items(self, item_ids, item_extra):
        return self.item_tower(
            self.input_dropout(
                torch.cat([self.item_embedding(item_ids), item_extra], dim=-1)
            )
        )

    def score_branches(self, user_branch, item_branch):
        return self.head(
            self.fusion_dropout(torch.cat([user_branch, item_branch], dim=-1))
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_branches(
            self.encode_users(user_ids, user_extra),
            self.encode_items(item_ids, item_extra),
        )


class LateFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        id_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.id_fusion = mlp([2 * embed_dim, *id_dims], dropout)
        self.head = mlp(
            [id_dims[-1] + user_extra_dim + item_extra_dim, *head_dims, 1],
            dropout,
            final_activation=False,
        )

    def fuse_embeddings(self, user_emb, item_emb):
        return self.id_fusion(
            self.input_dropout(torch.cat([user_emb, item_emb], dim=-1))
        )

    def score_fused(self, fused, user_extra, item_extra):
        return self.head(
            self.fusion_dropout(torch.cat([fused, user_extra, item_extra], dim=-1))
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_fused(
            self.fuse_embeddings(
                self.user_embedding(user_ids),
                self.item_embedding(item_ids),
            ),
            user_extra,
            item_extra,
        )


def build_model(trial):
    if trial == "early":
        return EarlyFusionNCF(
            n_users,
            n_items,
            EMBED_DIM,
            USER_EXTRA_DIM,
            ITEM_EXTRA_DIM,
            BRANCH_DIMS,
            EARLY_HEAD_DIMS,
            DROPOUT,
        ).to(DEVICE)
    return LateFusionNCF(
        n_users,
        n_items,
        EMBED_DIM,
        USER_EXTRA_DIM,
        ITEM_EXTRA_DIM,
        LATE_ID_DIMS,
        LATE_HEAD_DIMS,
        DROPOUT,
    ).to(DEVICE)

## 4. Training

Every epoch rebuilds the negative matrix from `EpochNegativeSampler`, then optimises BPR on `positive_score - negative_score` with AdamW. The epoch with the best tuning NDCG is kept (patience `3`), its checkpoint is restored, and a fresh model is refit for that many epochs on all training rows. Checkpoints, training logs and configuration are written under `artifacts/ncf_features_<trial>_<loss>/` for every trial and loss.


In [ ]:
def train_epoch(
    model,
    optimizer,
    user_ids,
    profile_idx,
    pos_rows,
    neg_matrix,
    neg_len,
    user_features,
    item_extra_tensor,
    rng,
):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(user_ids[batch], dtype=torch.long, device=DEVICE)
        user_extra = torch.as_tensor(
            user_features[profile_idx[batch]],
            dtype=torch.float32,
            device=DEVICE,
        )
        positives = torch.as_tensor(pos_rows[batch], dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(neg_matrix[batch], dtype=torch.long, device=DEVICE)
        neg_lens = torch.as_tensor(neg_len[batch], dtype=torch.long, device=DEVICE)
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :] < neg_lens[:, None]
        )
        users_expanded = users[:, None].expand(-1, negatives.shape[1])
        user_extra_expanded = (
            user_extra[:, None, :]
            .expand(-1, negatives.shape[1], -1)
            .reshape(-1, user_extra.shape[1])
        )
        negatives_flat = negatives.reshape(-1)
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model.score_pairs(
            users,
            positives,
            user_extra,
            item_extra_tensor[positives],
        )
        negative_scores = model.score_pairs(
            users_expanded.reshape(-1),
            negatives_flat,
            user_extra_expanded,
            item_extra_tensor[negatives_flat],
        ).reshape(len(batch), negatives.shape[1])
        if LOSS == "bpr":
            pair_loss = F.logsigmoid(positive_scores[:, None] - negative_scores)
            loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        else:
            scores = torch.cat([positive_scores, negative_scores[neg_mask]])
            labels = torch.cat(
                [
                    torch.ones_like(positive_scores),
                    torch.zeros_like(negative_scores[neg_mask]),
                ]
            )
            loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), SCORE_PAIRS):
            end = min(start + SCORE_PAIRS, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USER_IDS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEM_ROWS[start:end], dtype=torch.long, device=DEVICE
            )
            user_extra = torch.as_tensor(
                train_features[TUNE_PROFILE_IDX[start:end]],
                dtype=torch.float32,
                device=DEVICE,
            )
            scores[start:end] = (
                model.score_pairs(
                    users,
                    items,
                    user_extra,
                    ITEM_EXTRA_TENSOR[items],
                )
                .cpu()
                .numpy()
            )
    _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
    return ndcg


def train_model(model, trial):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)
    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        user_ids, profile_idx, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            user_ids,
            profile_idx,
            pos_rows,
            neg_matrix,
            neg_len,
            train_features,
            ITEM_EXTRA_TENSOR,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"[{trial}] epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print(f"[{trial}] Early stopping")
            break

    stopper.restore(model)
    print(
        f"[{trial}] Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = build_model(trial)
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)
    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        user_ids, profile_idx, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            user_ids,
            profile_idx,
            pos_rows,
            neg_matrix,
            neg_len,
            train_features,
            ITEM_EXTRA_TENSOR,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"[{trial}] refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")
    return final_model, training_log, stopper


def save_model_artifact(trial, model, training_log, stopper, artifact_dir, model_name):
    torch.save(
        {
            "state_dict": model.state_dict(),
            "category_vocab": CATEGORY_VOCAB,
            "config": {
                "trial": trial,
                "model": model_name,
                "n_users": n_users,
                "n_items": n_items,
                "embed_dim": EMBED_DIM,
                "user_extra_dim": USER_EXTRA_DIM,
                "item_extra_dim": ITEM_EXTRA_DIM,
                "category_dim": CATEGORY_DIM,
                "history_dim": HISTORY_DIM,
                "branch_dims": list(BRANCH_DIMS),
                "early_head_dims": list(EARLY_HEAD_DIMS),
                "late_id_dims": list(LATE_ID_DIMS),
                "late_head_dims": list(LATE_HEAD_DIMS),
                "dropout": DROPOUT,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": int(stopper.best_epoch),
                "tune_ndcg": float(stopper.best_score),
                "ndcg_k": int(TUNE_NDCG_K),
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "objective": LOSS,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "training_data": "labels_train full training pool",
            },
        },
        artifact_dir / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(artifact_dir / "training_log.csv")
    print(f"[{trial}] Saved model:", artifact_dir / "model.pt")


TRAINED_TRIALS = {}
for loss in RUN_LOSSES:
    LOSS = loss
    for trial in TRIALS:
        model_name = f"ncf_features_{trial}_{loss}"
        artifact_dir = ARTIFACTS / model_name
        artifact_dir.mkdir(parents=True, exist_ok=True)

        model_path = artifact_dir / "model.pt"
        expected_config = {
            "trial": trial,
            "n_users": n_users,
            "n_items": n_items,
            "embed_dim": EMBED_DIM,
            "user_extra_dim": USER_EXTRA_DIM,
            "item_extra_dim": ITEM_EXTRA_DIM,
            "branch_dims": list(BRANCH_DIMS),
            "dropout": DROPOUT,
            "objective": loss,
        }
        if REUSE_ARTIFACTS and model_path.exists():
            checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
            saved_config = checkpoint.get("config", {})
            if all(
                saved_config.get(key) == value
                for key, value in expected_config.items()
            ):
                model = build_model(trial)
                model.load_state_dict(checkpoint["state_dict"])
                model.eval()
                TRAINED_TRIALS[(loss, trial)] = {
                    "loss": loss,
                    "trial": trial,
                    "model": model,
                    "model_name": model_name,
                    "artifact_dir": artifact_dir,
                    "training_log": [],
                    "stopper": None,
                }
                print(f"[{loss}/{trial}] Loaded model: {model_path}")
                continue
            print(f"[{loss}/{trial}] Checkpoint config mismatch; retraining")

        model = build_model(trial)
        print(f"[{loss}/{trial}] Training {model_name}")
        model, training_log, stopper = train_model(model, trial)
        save_model_artifact(
            trial, model, training_log, stopper, artifact_dir, model_name
        )
        TRAINED_TRIALS[(loss, trial)] = {
            "loss": loss,
            "trial": trial,
            "model": model,
            "model_name": model_name,
            "artifact_dir": artifact_dir,
            "training_log": training_log,
            "stopper": stopper,
        }

## 5. Generate the standard candidate table

`select_top_k` scores each request against articles available by its timestamp, removes the request's history, ranks by score descending, one-hour-lagged click popularity descending, then ascending `news_code`, and backfills short lists from popularity. Early fusion precomputes the static item tower once and evaluates only the head per request-article pair; late fusion runs the ID-fusion layers for every pair. Requests with no history receive the popularity order directly. Rows are streamed through `CandidateWriter` in timestamp order to `artifacts/ncf_features_<trial>_<loss>/recommendations_<split>_top500.parquet`.


In [7]:
def popularity_select(order, excluded=None, k=CANDIDATE_CAP):
    excluded = set() if excluded is None else set(map(int, excluded))
    selected = [int(p) for p in order if int(p) not in excluded][:k]
    return np.asarray(selected, dtype=np.int64)


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(scores, popularity, news_code, k, excluded=None, fallback_order=None):
    scores = np.asarray(scores, dtype=np.float32)
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            ties = ties[np.lexsort((news_code[ties], -popularity[ties]))[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        selected = np.r_[
            selected,
            popularity_fill(remaining, popularity, news_code, k - selected.size),
        ]

    return selected

In [8]:
def encode_early_items(model, item_extra_tensor, batch=65_536):
    out = np.empty((n_items, model.branch_dim), dtype=np.float32)
    ids = torch.arange(n_items, dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        for start in range(0, n_items, batch):
            end = min(start + batch, n_items)
            out[start:end] = (
                model.encode_items(ids[start:end], item_extra_tensor[start:end])
                .cpu()
                .numpy()
            )
    return torch.from_numpy(out).to(DEVICE)


def score_early(model, user_branch, item_branch, max_pairs=SCORE_PAIRS):
    n_users = user_branch.shape[0]
    n_items_local = item_branch.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                rows = u1 - u0
                cols = i1 - i0
                user_expand = user_branch[u0:u1, None, :].expand(
                    rows, cols, model.branch_dim
                )
                item_expand = item_branch[None, i0:i1, :].expand(
                    rows, cols, model.branch_dim
                )
                scores[u0:u1, i0:i1] = (
                    model.score_branches(user_expand, item_expand).cpu().numpy()
                )
    return scores


def score_late(
    model,
    user_ids,
    user_extra,
    item_emb,
    item_extra,
    max_pairs=SCORE_PAIRS,
):
    n_users = len(user_ids)
    n_items_local = item_emb.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    user_emb = model.user_embedding(
        torch.as_tensor(user_ids, dtype=torch.long, device=DEVICE)
    )
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                rows = u1 - u0
                cols = i1 - i0
                fused = model.fuse_embeddings(
                    user_emb[u0:u1, None, :].expand(rows, cols, EMBED_DIM),
                    item_emb[None, i0:i1, :].expand(rows, cols, EMBED_DIM),
                )
                scores[u0:u1, i0:i1] = (
                    model.score_fused(
                        fused,
                        user_extra[u0:u1, None, :].expand(rows, cols, USER_EXTRA_DIM),
                        item_extra[None, i0:i1, :].expand(rows, cols, ITEM_EXTRA_DIM),
                    )
                    .cpu()
                    .numpy()
                )
    return scores


def score_target_split(trial, model, artifact_dir):
    recommendations_path = (
        artifact_dir / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
    )
    timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    profile_hours = hour_index_of(timestamps.astype(np.int64))
    history_rows = target_profiles["full_history_rows"].to_list()
    has_history = np.fromiter(
        (len(rows) > 0 for rows in history_rows),
        dtype=bool,
        count=len(history_rows),
    )
    impression_uids = target_profiles["impression_uid"].to_numpy()
    target_user_codes = target_profiles["user_code"].to_numpy().astype(np.int64)

    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))
    print(f"[{trial}] Scoring requests: {len(timestamps):,}")

    model.eval()
    sorted_ids = torch.as_tensor(ARTICLE_ROWS_SORTED, dtype=torch.long, device=DEVICE)
    item_branch_sorted = item_emb_sorted = item_extra_sorted = None
    if trial == "early":
        item_branch_sorted = encode_early_items(model, ITEM_EXTRA_TENSOR).index_select(
            0, sorted_ids
        )
    else:
        with torch.no_grad():
            item_emb_sorted = model.item_embedding(sorted_ids)
            item_extra_sorted = ITEM_EXTRA_TENSOR[sorted_ids]

    cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(recommendations_path)
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            batch_hour = int(profile_hours[start])
            n_eligible = int(np.searchsorted(AVAILABLE_SORTED, timestamp, side="right"))
            item_rows_batch = ARTICLE_ROWS_SORTED[:n_eligible]
            news_batch = NEWS_SORTED[:n_eligible]
            popularity_batch = hourly_popularity(batch_hour, item_rows_batch)
            fallback_order = np.lexsort((news_batch, -popularity_batch))

            user_ids = target_user_codes[start:end]
            user_extra = torch.as_tensor(
                target_features[start:end],
                dtype=torch.float32,
                device=DEVICE,
            )
            positions_tensor = torch.arange(n_eligible, dtype=torch.long, device=DEVICE)
            if trial == "early":
                with torch.no_grad():
                    user_branch = model.encode_users(
                        torch.as_tensor(user_ids, dtype=torch.long, device=DEVICE),
                        user_extra,
                    )
                scores = score_early(
                    model,
                    user_branch,
                    item_branch_sorted[positions_tensor],
                )
            else:
                scores = score_late(
                    model,
                    user_ids,
                    user_extra,
                    item_emb_sorted[positions_tensor],
                    item_extra_sorted[positions_tensor],
                )

            selected_batch = np.empty((end - start, CANDIDATE_CAP), dtype=np.int64)
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                if rows:
                    excluded = POS_IN_SORTED[np.asarray(rows, dtype=np.int64)]
                    excluded = excluded[(excluded >= 0) & (excluded < n_eligible)]
                else:
                    excluded = np.empty(0, dtype=np.int64)
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=(
                        None if has_history[profile_row] else fallback_order
                    ),
                )

            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"[{trial}] Scored {batch_no:,}/{len(batches):,} "
                    f"timestamp batches; {writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    print(f"[{trial}] Wrote candidates in {time.time() - started:.1f}s")
    return recommendations_path


for loss in RUN_LOSSES:
    LOSS = loss
    for trial in TRIALS:
        state = TRAINED_TRIALS[(loss, trial)]
        state["recommendations_path"] = score_target_split(
            trial, state["model"], state["artifact_dir"]
        )

[late] Scoring requests: 31,624
[late] Scored 1,000/24,580 timestamp batches; 523,500 rows
[late] Scored 2,000/24,580 timestamp batches; 1,099,000 rows
[late] Scored 3,000/24,580 timestamp batches; 1,718,000 rows
[late] Scored 4,000/24,580 timestamp batches; 2,367,000 rows
[late] Scored 5,000/24,580 timestamp batches; 3,048,000 rows
[late] Scored 6,000/24,580 timestamp batches; 3,734,000 rows
[late] Scored 7,000/24,580 timestamp batches; 4,404,000 rows
[late] Scored 8,000/24,580 timestamp batches; 5,095,500 rows
[late] Scored 9,000/24,580 timestamp batches; 5,771,000 rows
[late] Scored 10,000/24,580 timestamp batches; 6,465,500 rows
[late] Scored 11,000/24,580 timestamp batches; 7,188,000 rows
[late] Scored 12,000/24,580 timestamp batches; 7,901,000 rows
[late] Scored 13,000/24,580 timestamp batches; 8,620,500 rows
[late] Scored 14,000/24,580 timestamp batches; 9,331,000 rows
[late] Scored 15,000/24,580 timestamp batches; 10,035,500 rows
[late] Scored 16,000/24,580 timestamp batches; 1

## 6. Evaluation

Uses the shared `CandidateEvaluator` for every loss and trial and writes `results/candidate_generation_<model>_<split>.csv`. It reports recall (pooled and impression-mean), hit rate, long-tail recall, cold-article recall, novelty (bits), catalogue coverage and personalisation at each M. The final cell pivots the six headline metrics so early and late fusion can be compared side by side.


In [9]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    for trial in TRIALS:
        state = TRAINED_TRIALS[(loss, trial)]
        evaluator = CandidateEvaluator(root=DATA_ROOT, split=TARGET_SPLIT)
        evaluation = evaluator.evaluate_to_csv(
            state["recommendations_path"],
            model_name=state["model_name"],
            m_values=M_VALUES,
            output_dir=RESULTS,
        )
        print(f"[{loss}/{trial}]")
        print(evaluation.sort("m"))

[bpr/late]
shape: (32, 7)
┌────────────────────┬────────────┬─────┬───────────────────┬───────────┬────────────┬─────────────┐
│ model              ┆ split      ┆ m   ┆ metric            ┆ value     ┆ numerator  ┆ denominator │
│ ---                ┆ ---        ┆ --- ┆ ---               ┆ ---       ┆ ---        ┆ ---         │
│ str                ┆ str        ┆ i64 ┆ str               ┆ f64       ┆ f64        ┆ f64         │
╞════════════════════╪════════════╪═════╪═══════════════════╪═══════════╪════════════╪═════════════╡
│ ncf_features_late_ ┆ validation ┆ 50  ┆ catalogue_coverag ┆ 0.008057  ┆ 383.0      ┆ 47538.0     │
│ bpr                ┆            ┆     ┆ e                 ┆           ┆            ┆             │
│ ncf_features_late_ ┆ validation ┆ 50  ┆ cold_article_reca ┆ 0.0       ┆ 0.0        ┆ 342.0       │
│ bpr                ┆            ┆     ┆ ll                ┆           ┆            ┆             │
│ ncf_features_late_ ┆ validation ┆ 50  ┆ hit_rate          ┆ 0.0

In [10]:
comparison = pl.read_csv(COMPARISON_PATH)
model_names = [state["model_name"] for state in TRAINED_TRIALS.values()]
loss_table = (
    comparison.filter(pl.col("model").is_in(model_names))
    .filter(
        pl.col("metric").is_in(
            [
                "recall_pooled",
                "hit_rate",
                "catalogue_coverage",
                "personalisation",
                "long_tail_recall",
                "cold_article_recall",
            ]
        )
    )
    .pivot(on="model", index=["metric", "m"], values="value")
    .sort(["metric", "m"])
)
print(loss_table)

shape: (24, 4)
┌─────────────────────┬─────┬────────────────────────┬───────────────────────┐
│ metric              ┆ m   ┆ ncf_features_early_bpr ┆ ncf_features_late_bpr │
│ ---                 ┆ --- ┆ ---                    ┆ ---                   │
│ str                 ┆ i64 ┆ f64                    ┆ f64                   │
╞═════════════════════╪═════╪════════════════════════╪═══════════════════════╡
│ catalogue_coverage  ┆ 50  ┆ 0.008078               ┆ 0.008057              │
│ catalogue_coverage  ┆ 100 ┆ 0.015545               ┆ 0.01563               │
│ catalogue_coverage  ┆ 200 ┆ 0.028272               ┆ 0.028398              │
│ catalogue_coverage  ┆ 500 ┆ 0.059847               ┆ 0.060099              │
│ cold_article_recall ┆ 50  ┆ 0.0                    ┆ 0.0                   │
│ …                   ┆ …   ┆ …                      ┆ …                     │
│ personalisation     ┆ 500 ┆ 0.01469                ┆ 0.026522              │
│ recall_pooled       ┆ 50  ┆ 0.00913

## Notes and limitations

- Request profiles come from `full_history_rows`, so user features are causal and change as history grows within the target split.
- Popularity is read one hour behind the request and is used only for tie-breaking and the empty-history fallback, not as a model input.
- Early fusion precomputes the item tower because it depends only on static article features; late fusion must run the ID-fusion layers for every candidate pair, which is slower.
- BPR negatives are observed slate non-clicks, so the model inherits the logging system's exposure and position bias.
- Up to four exposed negatives are redrawn from the same slate each epoch, which controls popularity bias better than a fixed negative sample.
- Training uses a chronological 90% tuning slice, early stopping and a final refit on all training groups.
- The models are fitted once on the training split and never refreshed, so they cannot react to drift inside validation or test.
- The head output is a ranking score, not a calibrated click probability.
- Category and text features give cold articles some signal in the item tower, but their ID embeddings are untrained, so `cold_article_recall` remains limited.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
